In [ ]:
!pip install -q requests opencv-python numpy matplotlib boto3 qrcode pillow

In [ ]:
import cv2
import numpy as np
import requests
import json as _json

MODEL_SERVER   = 'http://yolo-onnx-predictor.ericsson-projects.svc.cluster.local:8888'
INFER_URL      = f'{MODEL_SERVER}/v2/models/yolo-onnx/infer'
INPUT_SIZE     = 640
CONF_THRESHOLD = 0.35
IOU_THRESHOLD  = 0.45
MIN_BOX_AREA   = 24 * 24

COCO_CLASSES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
    'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
    'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
    'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
    'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
    'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair',
    'couch','potted plant','bed','dining table','toilet','tv','laptop','mouse',
    'remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator',
    'book','clock','vase','scissors','teddy bear','hair drier','toothbrush'
]

rng = np.random.default_rng(42)
CLASS_COLORS = {i: tuple(int(c) for c in rng.integers(80, 255, 3)) for i in range(len(COCO_CLASSES))}

SESSION = requests.Session()

def preprocess(frame):
    orig_h, orig_w = frame.shape[:2]
    scale = min(INPUT_SIZE / orig_w, INPUT_SIZE / orig_h)
    new_w = int(round(orig_w * scale))
    new_h = int(round(orig_h * scale))

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((INPUT_SIZE, INPUT_SIZE, 3), 114, dtype=np.uint8)
    pad_x = (INPUT_SIZE - new_w) // 2
    pad_y = (INPUT_SIZE - new_h) // 2
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized

    img = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = np.expand_dims(img.transpose(2, 0, 1), 0)
    meta = {'scale': scale, 'pad_x': pad_x, 'pad_y': pad_y}
    return tensor, meta

def infer(tensor):
    global SESSION
    payload = {
        'inputs': [{
            'name': 'images',
            'shape': list(tensor.shape),
            'datatype': 'FP32',
            'data': tensor.flatten().tolist(),
        }]
    }
    for attempt in range(3):
        try:
            r = SESSION.post(INFER_URL, json=payload, timeout=30)
            r.raise_for_status()
            out = r.json()['outputs'][0]['data']
            return np.array(out, dtype=np.float32).reshape(1, 84, 8400)
        except (requests.ConnectionError, requests.exceptions.ChunkedEncodingError):
            # OVMS recycled the keep-alive connection — open a fresh one and retry
            SESSION = requests.Session()
            if attempt == 2:
                raise

def parse_detections(raw, meta, orig_h, orig_w):
    preds = raw[0].T
    class_ids = np.argmax(preds[:, 4:], axis=1)
    confs = preds[np.arange(len(class_ids)), 4 + class_ids]
    mask = confs > CONF_THRESHOLD
    preds, confs, class_ids = preds[mask], confs[mask], class_ids[mask]
    if len(preds) == 0:
        return []

    cx, cy, w, h = preds[:, :4].T
    x1 = (cx - w / 2 - meta['pad_x']) / meta['scale']
    y1 = (cy - h / 2 - meta['pad_y']) / meta['scale']
    x2 = (cx + w / 2 - meta['pad_x']) / meta['scale']
    y2 = (cy + h / 2 - meta['pad_y']) / meta['scale']

    x1 = np.clip(np.round(x1), 0, orig_w - 1).astype(int)
    y1 = np.clip(np.round(y1), 0, orig_h - 1).astype(int)
    x2 = np.clip(np.round(x2), 0, orig_w - 1).astype(int)
    y2 = np.clip(np.round(y2), 0, orig_h - 1).astype(int)

    widths = x2 - x1
    heights = y2 - y1
    areas = widths * heights
    mask = (widths > 0) & (heights > 0) & (areas >= MIN_BOX_AREA)
    x1, y1, x2, y2 = x1[mask], y1[mask], x2[mask], y2[mask]
    widths, heights = widths[mask], heights[mask]
    confs, class_ids = confs[mask], class_ids[mask]
    if len(confs) == 0:
        return []

    detections = []
    for cid in np.unique(class_ids):
        cls_mask = class_ids == cid
        cls_boxes = list(zip(x1[cls_mask].tolist(), y1[cls_mask].tolist(), widths[cls_mask].tolist(), heights[cls_mask].tolist()))
        cls_scores = confs[cls_mask].tolist()
        indices = cv2.dnn.NMSBoxes(cls_boxes, cls_scores, CONF_THRESHOLD, IOU_THRESHOLD)
        if len(indices) == 0:
            continue
        cls_x1, cls_y1, cls_x2, cls_y2 = x1[cls_mask], y1[cls_mask], x2[cls_mask], y2[cls_mask]
        cls_confs = confs[cls_mask]
        for idx in np.array(indices).flatten():
            detections.append((cls_x1[idx], cls_y1[idx], cls_x2[idx], cls_y2[idx], cls_confs[idx], int(cid)))

    detections.sort(key=lambda det: det[4], reverse=True)
    return detections

def draw_boxes(frame, detections):
    for (x1, y1, x2, y2, conf, cid) in detections:
        color = CLASS_COLORS[cid]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f"{COCO_CLASSES[cid]} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(frame, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return frame

print(f"Ready. Sending frames to: {INFER_URL}")
print("Model classes: COCO 80-class label set.")
print("Demo note: COCO has no 'rabbit' class, so a rabbit may be mapped to the closest supported animal label.")

In [ ]:
import os
import subprocess
from pathlib import Path

cwd = Path.cwd()
candidates = [
    cwd / 'site_camera_01_mjpeg.avi',
    cwd / '5g-vision/site_camera_01_mjpeg.avi',
    cwd.parent / '5g-vision/site_camera_01_mjpeg.avi',
]
input_path = next((p for p in candidates if p.exists()), None)
if input_path is None:
    raise FileNotFoundError(f"Could not find site_camera_01_mjpeg.avi near cwd={cwd}")

INPUT_VIDEO      = str(input_path)
TMP_VIDEO        = 'site_camera_01_annotated_tmp.avi'
OUTPUT_VIDEO     = 'site_camera_01_annotated.webm'
OVERWRITE_OUTPUT = True

if OVERWRITE_OUTPUT:
    for path in (TMP_VIDEO, OUTPUT_VIDEO):
        if os.path.exists(path):
            os.remove(path)

cap    = cv2.VideoCapture(INPUT_VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS) or 25
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Input: {width}x{height} @ {fps:.1f}fps — {total} frames")

# Write annotated frames as MJPEG AVI (cv2 built-in encoder, no FFmpeg needed)
out = cv2.VideoWriter(TMP_VIDEO, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))

sample_frames = []
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    tensor, meta = preprocess(frame)
    detections = parse_detections(infer(tensor), meta, height, width)
    annotated = draw_boxes(frame.copy(), detections)
    out.write(annotated)
    if frame_idx % max(1, total // 6) == 0:
        sample_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    frame_idx += 1
    print(f"  {frame_idx}/{total}", end='\r')

cap.release()
out.release()
print(f"\nAnnotated frames written to '{TMP_VIDEO}'")

# Transcode to VP9 WebM — plays in Brave/Chrome/Firefox/Safari
# System FFmpeg has libvpx-vp9 encoder enabled even without H.264
print("Transcoding to VP9 WebM for browser compatibility...")
subprocess.run([
    'ffmpeg', '-y', '-i', TMP_VIDEO,
    '-c:v', 'libvpx-vp9', '-crf', '33', '-b:v', '0',
    OUTPUT_VIDEO
], check=True, capture_output=True)
os.remove(TMP_VIDEO)
print(f"Saved -> '{OUTPUT_VIDEO}'")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import base64

# If inference was skipped, sample frames from the existing annotated video
if 'sample_frames' not in dir() or not sample_frames:
    sample_frames = []
    cap_out = cv2.VideoCapture(OUTPUT_VIDEO)
    total_out = int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT))
    for idx in range(6):
        cap_out.set(cv2.CAP_PROP_POS_FRAMES, idx * max(1, total_out // 6))
        ret, frame = cap_out.read()
        if ret:
            sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap_out.release()

# Sample frame grid
n = len(sample_frames)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
for ax, img in zip([axes] if n == 1 else axes, sample_frames):
    ax.imshow(img)
    ax.axis('off')
fig.suptitle('5G Site Camera — YOLOv8 Detections', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Inline video
with open(OUTPUT_VIDEO, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(f"""
<h3>▶ Annotated Site Camera Feed</h3>
<video width="720" controls loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
"""))

In [ ]:
import boto3
import os
import qrcode
from botocore.client import Config
from IPython.display import HTML, display

S3_ENDPOINT   = 'http://minio-fallback.aistor.svc.cluster.local:9000'
S3_ACCESS_KEY = 'minioadm'
S3_SECRET_KEY = 'minioadm'
bucket        = 'vision-models'
s3_key        = 'output/site_camera_01_annotated.webm'

# Public route — update if the cluster is rebuilt (oc get route minio-download -n aistor)
MINIO_PUBLIC_ENDPOINT = 'https://minio-download-aistor.apps.cluster-ls8dt.ls8dt.sandbox1882.opentlc.com'

s3_internal = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

print(f"Uploading '{OUTPUT_VIDEO}' -> s3://{bucket}/{s3_key} ...")
s3_internal.upload_file(OUTPUT_VIDEO, bucket, s3_key,
                        ExtraArgs={'ContentType': 'video/webm'})
print("Upload complete.\n")

# Presigned URL via the public route (valid 24 h)
s3_public = boto3.client(
    's3',
    endpoint_url=MINIO_PUBLIC_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)
presigned = s3_public.generate_presigned_url(
    'get_object',
    Params={'Bucket': bucket, 'Key': s3_key},
    ExpiresIn=86400,
)

local_path = os.path.abspath(OUTPUT_VIDEO)
print('=' * 60)
print('Local file (download via JupyterLab file browser):')
print(f'  {local_path}')
print()
print('Presigned URL (24 h):')
print(f'  {presigned}')
print('=' * 60)

qr = qrcode.QRCode(box_size=8, border=2)
qr.add_data(presigned)
qr.make(fit=True)
qr_img = qr.make_image(fill_color='black', back_color='white')

display(HTML('<h3>Scan To Open The Annotated Video</h3>'))
display(qr_img)